# Arabic Text Error Detection, Correction & Morphological Analysis

This notebook takes Arabic text and runs it through a pipeline that:

1. **Verifies** the input is actually Arabic
2. **Tokenizes** the text
3. **Detects spelling errors** (words with no valid morphological analysis)
4. **Detects grammatical errors** (noun–adjective agreement mismatches)
5. **Color-codes** the detected errors directly on the original text
6. **Corrects spelling errors** (frequency-ranked, dictionary-validated candidates)
7. **Corrects grammatical errors** (morphological reinflection, not text generation)
8. **Normalizes** each word (orthographic canonicalization)
9. **Stems** each word (light stemming)
10. **Lemmatizes** each word (dictionary/citation form)

**No LLMs are used anywhere in this pipeline.** Every step is a classical,
inspectable NLP technique: a finite-state/DB-backed morphological analyzer,
a maximum-likelihood disambiguator, an edit-distance spell checker ranked by
corpus frequency, and rule-based agreement checking. You can inspect exactly
*why* the notebook flagged or changed anything.

## Tools used (all open source, all researched on GitHub / PyPI / Hugging Face)

| Task | Tool | Source |
|---|---|---|
| Morphological analysis, disambiguation, lemmatization, reinflection | **CAMeL Tools** (NYU Abu Dhabi CAMeL Lab) | github.com/CAMeL-Lab/camel_tools |
| Spelling-candidate ranking | **SymSpell** (`symspellpy`) + **FrequencyWords** Arabic 50k list | github.com/wolfgarbe/SymSpell, github.com/hermitdave/FrequencyWords |
| Light stemming / root extraction | **Tashaphyne** | github.com/linuxscout/tashaphyne |
| Extra normalization helpers | **PyArabic** | github.com/linuxscout/pyarabic |
| Language identification (secondary check) | **langdetect** | github.com/fedelopez77/langdetect (pip: `langdetect`) |

## How to use this notebook

Run the cells **top to bottom once** (Setup takes a couple of minutes — it downloads
~130MB of morphology data). After that, every step has its own cell with its own
printed output, so you can re-run any single step on its own. The **"Full pipeline"**
section near the end ties every step into one function, and the **"Try it with your
own text"** section lets you run that function on anything you type, without
re-running the setup.

> **License note:** CAMeL Tools, PyArabic, and symspellpy are MIT/permissive.
> Tashaphyne is GPL-3.0 — worth knowing if you plan to redistribute this as
> part of a closed-source product.

## 0. Setup

Installs the libraries, downloads the two CAMeL Tools data packages this
notebook actually needs (the morphological analysis database and the MLE
disambiguation model — about 130MB total), and downloads a 50k-word Arabic
frequency list used later for spelling suggestions.

In [1]:
# Core Arabic NLP toolkit (morphology, disambiguation, lemmatization, reinflection)
# + spelling (symspellpy), stemming (tashaphyne, pulls in pyarabic), language ID (langdetect)
!pip install -q camel-tools symspellpy tashaphyne langdetect pandas

import sys
print(f"Python: {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("camel-tools needs Python 3.11+. In Colab: Runtime > Change runtime type.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [2]:
# Download only the two CAMeL Tools data packages this notebook uses:
#  - morphology-db-msa-r13   : the morphological analysis/generation database (Modern Standard Arabic)
#  - disambig-mle-calima-msa-r13 : the pretrained MLE disambiguation model (POS, gender, number, lemma...)
# (There's also a bundled `camel_data -i light` shortcut, but as of this writing it also pulls in
#  dialect-ID models and three dialect morphology DBs -- 350MB+ this notebook never uses -- so we
#  name the two packages explicitly to keep setup fast.)
!camel_data -i morphology-db-msa-r13
!camel_data -i disambig-mle-calima-msa-r13

The following packages will be installed: 'morphology-db-msa-r13'
Extracting package 'morphology-db-msa-r13': 100% 40.5M/40.5M [00:00<00:00, 352MB/s]
The following packages will be installed: 'disambig-mle-calima-msa-r13'
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:00<00:00, 219MB/s]


In [3]:
# A 50,000-word Arabic frequency list (word + corpus count per line), used to rank
# spelling-correction candidates. Source: hermitdave/FrequencyWords on GitHub, built
# from OpenSubtitles data -- a standard, widely-used resource for this exact purpose.
!wget -q -O ar_50k.txt "https://raw.githubusercontent.com/hermitdave/FrequencyWords/master/content/2016/ar/ar_50k.txt"
!wc -l ar_50k.txt

50000 ar_50k.txt


In [4]:
import re
import html
import pandas as pd
from IPython.display import display, HTML

from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.morphology.generator import Generator
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (
    normalize_alef_ar, normalize_alef_maksura_ar, normalize_teh_marbuta_ar,
)
import pyarabic.araby as araby
from tashaphyne.stemming import ArabicLightStemmer
from symspellpy import SymSpell, Verbosity
from langdetect import detect_langs, DetectorFactory, LangDetectException
DetectorFactory.seed = 0  # deterministic langdetect results

print("Loading morphological analyzer ...")
analyzer = Analyzer(MorphologyDB.builtin_db())

print("Loading morphological generator ...")
generator = Generator(MorphologyDB.builtin_db(flags='g'))

print("Loading MLE disambiguator (POS / gender / number / lemma) ...")
mle_disambiguator = MLEDisambiguator.pretrained()

print("Loading spelling frequency dictionary ...")
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary('ar_50k.txt', term_index=0, count_index=1, separator=' ', encoding='utf-8')

light_stemmer = ArabicLightStemmer()

print("Ready.")

Loading morphological analyzer ...
Loading morphological generator ...
Loading MLE disambiguator (POS / gender / number / lemma) ...
Loading spelling frequency dictionary ...
Ready.


## Sample text

This is the running example used to demonstrate every step below. It
deliberately contains: a spelling typo (**المدرصة**), a gender-agreement
error (**الطالبة الجديد** — feminine noun, masculine adjective), and a
diacritized word (**صباحاً**) to show normalization actually doing something.
Two clean, correct sentences follow it so you can see the tool leave correct
text alone.

Edit `input_text` and re-run the notebook (or just re-run the "Full pipeline"
cell later) to try something else.

In [139]:
input_text = "ذهبت الطالبة الجديد إلى المدرصة ضباحاً. هي تحبان القراءة كتيراً، ولديها كتاب جميلان تقرأه كل يوم."
print(input_text)

ذهبت الطالبة الجديد إلى المدرصة ضباحاً. هي تحبان القراءة كتيراً، ولديها كتاب جميلان تقرأه كل يوم.


## Step 1 — Verify the text is Arabic

Two independent signals, combined:

1. **Unicode-range ratio** — what fraction of the *letters* in the text fall
   in the Arabic Unicode blocks. This is deterministic and instant, and is
   the primary signal.
2. **`langdetect`** — a statistical language-ID model, used as a secondary,
   informational cross-check.

`langdetect` alone is not reliable enough to be the primary check here: it's
trained on whole documents, so on short or mixed-script input it can confuse
Arabic with other Arabic-script languages (Urdu, Persian). Try it — a short
mixed Arabic/English string below gets a ~0.6 Arabic-character ratio but
`langdetect` reports Urdu. The ratio-based check is what actually gates the
rest of the pipeline; `langdetect`'s guess is reported for reference only.

In [132]:
ARABIC_RE = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]')

def arabic_char_ratio(text):
    # Fraction of alphabetic characters that fall in an Arabic Unicode block.
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    arabic_letters = [c for c in letters if ARABIC_RE.match(c)]
    return len(arabic_letters) / len(letters)

def check_is_arabic(text, ratio_threshold=0.6):
    ratio = arabic_char_ratio(text)
    try:
        top = detect_langs(text)[0]
        ld_lang, ld_conf = top.lang, round(top.prob, 3)
    except LangDetectException:
        ld_lang, ld_conf = None, None
    return {
        'is_arabic': ratio >= ratio_threshold,
        'arabic_char_ratio': round(ratio, 3),
        'langdetect_lang': ld_lang,
        'langdetect_confidence': ld_conf,
    }

# Demo on the sample text, plus two contrast cases
for sample in [input_text, "This is plain English.", "مرحبا hello مزيج نص"]:
    result = check_is_arabic(sample)
    label = "Arabic" if result['is_arabic'] else "NOT Arabic"
    print(f"[{label:10s}] ratio={result['arabic_char_ratio']:.0%}  "
          f"langdetect={result['langdetect_lang']} ({result['langdetect_confidence']})  "
          f"-> {sample[:40]!r}")

arabic_check = check_is_arabic(input_text)
assert arabic_check['is_arabic'], "Input does not look like Arabic -- stopping."

[Arabic    ] ratio=100%  langdetect=ar (1.0)  -> 'ذهبت الطالبة الجديد إلى المدرصة ضباحاً. '
[NOT Arabic] ratio=0%  langdetect=en (1.0)  -> 'This is plain English.'
[Arabic    ] ratio=69%  langdetect=fa (0.571)  -> 'مرحبا hello مزيج نص'


## Step 2 — Tokenization

Splits the text into words and punctuation using CAMeL Tools' whitespace/
punctuation tokenizer, and defines `is_arabic_token`, a small helper used by
every later step to skip punctuation, digits, and non-Arabic words so they
pass through unchanged.

One subtlety worth calling out: Python's `str.isalpha()` returns `False` for
a word that has Arabic *diacritics* attached (the short-vowel marks are
Unicode "combining marks", not "letters"), which would silently exclude
vocalized words like **صباحاً** from every later step. `is_arabic_token`
checks for the presence of Arabic base letters instead, so diacritized words
are still processed correctly.

In [140]:
ARABIC_LETTERS_RE = re.compile(r'[\u0621-\u063A\u0641-\u064A\u066E\u066F\u0671-\u06D3\u06D5]')
LATIN_RE = re.compile(r'[A-Za-z]')

def is_arabic_token(tok):
    return bool(ARABIC_LETTERS_RE.search(tok)) and not LATIN_RE.search(tok) and not tok.isdigit()

def tokenize_arabic(text):
    return simple_word_tokenize(text)

tokens = tokenize_arabic(input_text)
print(tokens)
print(f"\n{len(tokens)} tokens, {sum(is_arabic_token(t) for t in tokens)} of which are Arabic words")

['ذهبت', 'الطالبة', 'الجديد', 'إلى', 'المدرصة', 'ضباحاً', '.', 'هي', 'تحبان', 'القراءة', 'كتيراً', '،', 'ولديها', 'كتاب', 'جميلان', 'تقرأه', 'كل', 'يوم', '.']

19 tokens, 16 of which are Arabic words


## Step 3 — Spelling error detection

For each Arabic token, CAMeL Tools' morphological **Analyzer** is asked for
every possible morphological analysis of that surface form (it checks the
word against a large lexical database, trying every combination of prefixes,
stems, and suffixes it knows about). If **zero** analyses come back, the word
doesn't match anything the analyzer recognizes as Arabic morphology — a
strong, well-established signal of a spelling mistake (this is the same
zero-analysis signal the `camel_morphology` command-line tool reports as
`NO_ANALYSIS`).

This is more reliable than a plain dictionary lookup because it accounts for
Arabic's rich inflectional morphology (prefixes, suffixes, broken plurals,
verb conjugations, ...) rather than only matching bare dictionary headwords.

In [141]:
def detect_spelling_errors(tokens):
    # Returns {token: [positions]} for tokens with zero morphological analyses.
    errors = {}
    for i, tok in enumerate(tokens):
        if not is_arabic_token(tok):
            continue
        if len(analyzer.analyze(tok)) == 0:
            errors.setdefault(tok, []).append(i)
    return errors

spelling_errors = detect_spelling_errors(tokens)

print(f"{len(spelling_errors)} candidate spelling error(s) found:")
for word, positions in spelling_errors.items():
    print(f"  - '{word}'  at position(s) {positions}")

3 candidate spelling error(s) found:
  - 'المدرصة'  at position(s) [4]
  - 'ضباحاً'  at position(s) [5]
  - 'كتيراً'  at position(s) [10]


## Step 4 — Grammatical error detection

CAMeL Tools' **MLE disambiguator** assigns each word its most likely
part-of-speech, gender, number, state (definite/indefinite), case, and lemma
(each word is scored independently, so a nearby spelling mistake doesn't
throw off a word's own analysis — you can confirm this by re-running Step 4
before Step 3's correction is ever applied). This gives us structured
grammatical features to check agreement rules against.

**Rule implemented: noun–adjective gender/number agreement**, restricted on
purpose to **singular and dual nouns** immediately followed by an adjective.
Arabic plural agreement is *not* a simple "match the plural" rule — inanimate
("irrational") broken plurals grammatically take **feminine-singular**
adjective agreement (e.g. "الكتب **الجديدة**", lit. "the books the-new-FEM-SG",
is *correct* Arabic), while human plurals need real plural agreement. Getting
this rule wrong would produce confidently-wrong corrections, so plural nouns
are intentionally left unchecked — see the Notes section at the end.

Words already flagged as spelling errors are skipped here too, since their
grammatical features aren't trustworthy in the first place.

In [142]:
GENDER_LABEL = {'m': 'masculine', 'f': 'feminine'}

def analyze_words_in_context(tokens):
    # Runs the MLE disambiguator and returns one info-dict per token.
    disambiguated = mle_disambiguator.disambiguate(tokens)
    infos = []
    for i, dw in enumerate(disambiguated):
        if dw.analyses:
            a = dw.analyses[0].analysis
            infos.append({
                'index': i, 'word': dw.word, 'pos': a.get('pos'), 'gen': a.get('gen'),
                'num': a.get('num'), 'stt': a.get('stt'), 'cas': a.get('cas'),
                'lex': a.get('lex'), 'prc0': a.get('prc0'),
            })
        else:
            infos.append({'index': i, 'word': dw.word, 'pos': None})
    return infos

def detect_grammar_errors(word_infos, spelling_error_positions):
    issues = []
    for i in range(len(word_infos) - 1):
        current_word, next_word = word_infos[i], word_infos[i + 1]

        # Skip if either word is a spelling error
        if current_word['index'] in spelling_error_positions or next_word['index'] in spelling_error_positions:
            continue

        # --- Noun-Adjective Agreement Check ---
        is_noun_adj_candidate = False
        adj_for_check_info = None

        # Case 1: Standard noun followed by adjective
        if current_word.get('pos') == 'noun' and next_word.get('pos') == 'adj':
            is_noun_adj_candidate = True
            adj_for_check_info = next_word
        # Case 2: Noun followed by another word disambiguated as a noun, but whose lemma is an adjective
        # This handles cases like 'كتاب جميلان' where 'جميلان' (dual of 'جميل') is a noun.
        elif current_word.get('pos') == 'noun' and next_word.get('pos') == 'noun' and next_word.get('lex'):
            lemma_analyses = analyzer.analyze(next_word['lex'])
            # Check if any analysis of the lemma itself indicates it can be an adjective
            if any(analysis.get('pos') == 'adj' for analysis in lemma_analyses):
                is_noun_adj_candidate = True
                adj_for_check_info = next_word # We'll use its features for comparison

        if is_noun_adj_candidate and adj_for_check_info:
            if current_word.get('num') not in ('s', 'd'): # plural intentionally out of scope
                continue

            gender_mismatch = (current_word['gen'] != adj_for_check_info['gen'] and
                               'na' not in (current_word['gen'], adj_for_check_info['gen']))
            number_mismatch = (current_word['num'] != adj_for_check_info['num'] and
                               'na' not in (current_word['num'], adj_for_check_info['num']))

            if gender_mismatch or number_mismatch:
                parts = []
                if gender_mismatch:
                    parts.append(f"gender ({GENDER_LABEL.get(current_word['gen'], current_word['gen'])} noun vs "
                                  f"{GENDER_LABEL.get(adj_for_check_info['gen'], adj_for_check_info['gen'])} adjective)")
                if number_mismatch:
                    parts.append(f"number (noun is {current_word['num']}, adjective is {adj_for_check_info['num']})")
                issues.append({
                    'type': 'noun_adj',
                    'noun_idx': current_word['index'],
                    'adj_idx': next_word['index'], # Always use next_word's index for correction
                    'noun': current_word['word'],
                    'adj': next_word['word'],
                    'target_gen': current_word['gen'],
                    'target_num': current_word['num'],
                    'explanation': ' and '.join(parts),
                })

        # --- Subject-Verb Agreement Check ---
        if current_word.get('pos') in ('pron', 'noun') and next_word.get('pos') == 'verb':
            # This is a simplified subject-verb agreement check for immediately adjacent words.
            subject = current_word
            verb = next_word

            gender_mismatch = (subject['gen'] != verb['gen'] and
                               'na' not in (subject['gen'], verb['gen']))
            number_mismatch = (subject['num'] != verb['num'] and
                               'na' not in (subject['num'], verb['num']))

            if gender_mismatch or number_mismatch:
                parts = []
                if gender_mismatch:
                    parts.append(f"gender ({GENDER_LABEL.get(subject['gen'], subject['gen'])} subject vs "
                                  f"{GENDER_LABEL.get(verb['gen'], verb['gen'])} verb)")
                if number_mismatch:
                    parts.append(f"number (subject is {subject['num']}, verb is {verb['num']})")
                issues.append({
                    'type': 'sub_verb',
                    'subject_idx': subject['index'],
                    'verb_idx': verb['index'],
                    'subject': subject['word'],
                    'verb': verb['word'],
                    'target_gen': subject['gen'],
                    'target_num': subject['num'],
                    'explanation': ' and '.join(parts),
                })
    return issues

word_infos = analyze_words_in_context(tokens)
spelling_error_positions = {p for positions in spelling_errors.values() for p in positions}
grammar_issues = detect_grammar_errors(word_infos, spelling_error_positions)

print(f"{len(grammar_issues)} grammatical agreement issue(s) found:")
for issue in grammar_issues:
    if issue['type'] == 'noun_adj':
        print(f"  - '{issue['noun']}' + '{issue['adj']}'  ->  mismatch in {issue['explanation']}")
    elif issue['type'] == 'sub_verb':
        print(f"  - '{issue['subject']}' + '{issue['verb']}'  ->  subject-verb mismatch in {issue['explanation']}")

3 grammatical agreement issue(s) found:
  - 'الطالبة' + 'الجديد'  ->  mismatch in gender (feminine noun vs masculine adjective)
  - 'هي' + 'تحبان'  ->  subject-verb mismatch in number (subject is s, verb is d)
  - 'جميلان' + 'تقرأه'  ->  subject-verb mismatch in gender (masculine subject vs feminine verb)


To debug why `كتاب جميلان` is not being flagged, let's examine the detailed morphological analysis for these two words from `word_infos`.

In [30]:
print(f"Analysis for 'كتاب' (index 13): {word_infos[13]}")
print(f"Analysis for 'جميلان' (index 14): {word_infos[14]}")

Analysis for 'كتاب' (index 13): {'index': 13, 'word': 'كتاب', 'pos': 'noun', 'gen': 'm', 'num': 's', 'stt': 'c', 'cas': 'g', 'lex': 'كِتاب', 'prc0': '0'}
Analysis for 'جميلان' (index 14): {'index': 14, 'word': 'جميلان', 'pos': 'noun', 'gen': 'm', 'num': 'd', 'stt': 'i', 'cas': 'n', 'lex': 'جَمِيل', 'prc0': '0'}


The MLE disambiguator assigned `جميلان` as a noun. Let's see what other morphological analyses CAMeL Tools' `analyzer` gives for `جميلان` to confirm if 'adj' is even a possible part of speech.

In [89]:
print(analyzer.analyze('كتاب'))
print(analyzer.analyze('جميلان'))

[{'diac': 'كُتّاب', 'lex': 'كُتّاب', 'bw': 'كُتّاب/NOUN', 'gloss': 'kuttab_(village_school);Quran_school', 'pos': 'noun', 'prc3': '0', 'prc2': '0', 'prc1': '0', 'prc0': '0', 'per': 'na', 'asp': 'na', 'vox': 'na', 'mod': 'na', 'stt': 'i', 'cas': 'u', 'enc0': '0', 'rat': 'i', 'source': 'lex', 'form_gen': 'm', 'form_num': 's', 'd3seg': 'كُتّاب', 'caphi': 'k_u_t_t_aa_b', 'd1tok': 'كُتّاب', 'd2tok': 'كُتّاب', 'pos_logprob': -0.4344233, 'd3tok': 'كُتّاب', 'd2seg': 'كُتّاب', 'pos_lex_logprob': -99.0, 'num': 's', 'ud': 'NOUN', 'gen': 'm', 'catib6': 'NOM', 'root': 'ك.ت.ب', 'bwtok': 'كُتّاب', 'pattern': '1ُ2ّا3', 'lex_logprob': -99.0, 'atbtok': 'كُتّاب', 'atbseg': 'كُتّاب', 'd1seg': 'كُتّاب', 'stem': 'كُتّاب', 'stemgloss': 'kuttab_(village_school);Quran_school', 'stemcat': 'N'}, {'diac': 'كُتّابَ', 'lex': 'كُتّاب', 'bw': 'كُتّاب/NOUN+َ/CASE_DEF_ACC', 'gloss': 'kuttab_(village_school);Quran_school+[def.acc.]', 'pos': 'noun', 'prc3': '0', 'prc2': '0', 'prc1': '0', 'prc0': '0', 'per': 'na', 'asp': 

## Step 5 — Color-code the detected errors

Renders the original sentence with spelling errors highlighted in **red**
and grammar errors highlighted in **orange**, each with a hover tooltip
explaining the issue. This runs on the *original* text, before any
correction — it's a picture of what Steps 3 and 4 found.

In [69]:
def render_highlighted_html(tokens, spelling_errors, grammar_issues):
    spelling_positions = {p for positions in spelling_errors.values() for p in positions}
    grammar_positions = {}
    for issue in grammar_issues:
        if issue['type'] == 'noun_adj':
            grammar_positions[issue['adj_idx']] = issue['explanation']
        elif issue['type'] == 'sub_verb':
            grammar_positions[issue['verb_idx']] = issue['explanation']

    spans = []
    for i, tok in enumerate(tokens):
        safe_tok = html.escape(tok)
        if i in spelling_positions:
            spans.append(
                f'<span style="background:#ffd6d6;border-bottom:2px solid #d33;'
                f'border-radius:3px;padding:1px 4px;" title="Spelling error">{safe_tok}</span>'
            )
        elif i in grammar_positions:
            spans.append(
                f'<span style="background:#ffe8c2;border-bottom:2px solid #e08a00;'
                f'border-radius:3px;padding:1px 4px;" '
                f'title="Grammar error: {html.escape(grammar_positions[i])}">{safe_tok}</span>'
            )
        else:
            spans.append(safe_tok)

    legend = (
        '<div style="margin-bottom:8px;font-family:sans-serif;font-size:13px;color:#444;">'
        '<span style="background:#ffd6d6;border-bottom:2px solid #d33;padding:1px 6px;'
        'border-radius:3px;">spelling error</span>&nbsp;&nbsp;'
        '<span style="background:#ffe8c2;border-bottom:2px solid #e08a00;padding:1px 6px;'
        'border-radius:3px;">grammar error</span></div>'
    )
    body = (
        f'<div dir="rtl" style="font-size:22px;line-height:2.4;'
        f'font-family:\'Traditional Arabic\',Tahoma,Arial,sans-serif;">{ " ".join(spans)}</div>'
    )
    display(HTML(legend + body))

## Step 6 — Spelling error correction

For each flagged word, **SymSpell** generates candidate corrections within
edit-distance 2, ranked by how often each candidate appears in a 50k-word
Arabic frequency corpus. Those candidates are then filtered through the same
morphological **Analyzer** from Step 3, keeping only candidates that are
themselves valid Arabic words.

That filter matters: a frequency list alone can rank a *different real word*
above the one the writer actually meant (frequency lists are corpora, not
oracles), and — since informal Arabic corpora are full of nonstandard
spellings — can also rank informally-spelled non-words highly. Requiring the
morphological analyzer's approval on top of frequency ranking is the more
reliable combination.

In [143]:
def correct_spelling(tokens, spelling_errors):
    corrections = {}
    for word in spelling_errors:
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        valid = [s.term for s in suggestions if analyzer.analyze(s.term)]
        if valid:
            corrections[word] = valid[0]
        elif suggestions:
            corrections[word] = suggestions[0].term  # no analyzer-valid candidate; best-effort fallback
        else:
            corrections[word] = None  # nothing found; leave the original word as-is

    corrected_tokens = [
        corrections[tok] if (tok in corrections and corrections[tok]) else tok
        for tok in tokens
    ]
    return corrected_tokens, corrections

spelling_corrected_tokens, spelling_corrections = correct_spelling(tokens, spelling_errors)

for original, fixed in spelling_corrections.items():
    print(f"  '{original}'  ->  '{fixed}'")
print("\nText after spelling correction:")
print(' '.join(spelling_corrected_tokens))

  'المدرصة'  ->  'المدرسة'
  'ضباحاً'  ->  'صباحا'
  'كتيراً'  ->  'كثيرا'

Text after spelling correction:
ذهبت الطالبة الجديد إلى المدرسة صباحا . هي تحبان القراءة كثيرا ، ولديها كتاب جميلان تقرأه كل يوم .


## Step 7 — Grammatical error correction

This is genuine morphological **reinflection**, not text generation: for
each flagged adjective, CAMeL Tools' **Generator** is given the adjective's
own lemma plus a corrected feature set (the noun's gender/number, but the
*adjective's own* case, definiteness state, and determiner clitic, so only
the mismatched feature actually changes) and asked to produce the matching
surface form. If generation fails for a given word (rare, but possible for
irregular forms), the issue is still reported — it's just not silently
auto-fixed.

In [144]:
def correct_grammar(tokens, word_infos, grammar_issues):
    corrected = list(tokens)
    reports = []
    infos_by_index = {w['index']: w for w in word_infos}

    for issue in grammar_issues:
        if issue['type'] == 'noun_adj':
            target_word_info = infos_by_index[issue['adj_idx']]
            target_pos = 'adj' # We're reinflecting as an adjective
            target_gen = issue['target_gen']
            target_num = issue['target_num']
            lemma_to_generate = target_word_info.get('lex')
            original_word_idx = issue['adj_idx']
        elif issue['type'] == 'sub_verb':
            target_word_info = infos_by_index[issue['verb_idx']]
            target_pos = 'verb' # We're reinflecting as a verb
            target_gen = issue['target_gen'] # Subject's gender
            target_num = issue['target_num'] # Subject's number
            lemma_to_generate = target_word_info.get('lex')
            original_word_idx = issue['verb_idx']
        else:
            # Should not happen with defined types, but good practice
            reports.append((tokens[issue['adj_idx']], None, False)) # Fallback if issue type is unknown
            continue

        feats = {'pos': target_pos, 'gen': target_gen, 'num': target_num}
        # Copy other features from the original word, as they should be preserved if not mismatched
        for key in ('stt', 'cas'): # 'stt' and 'cas' are mostly for nouns/adjectives
            if key in target_word_info and target_word_info.get(key) not in (None, 'na', 'u'):
                feats[key] = target_word_info[key]
        if target_word_info.get('prc0') not in (None, '0', 'na'):
            feats['prc0'] = target_word_info['prc0']

        generated = []
        if lemma_to_generate:
            try:
                generated = generator.generate(lemma_to_generate, feats)
            except Exception:
                generated = []

        if generated:
            fixed_word = dediac_ar(generated[0]['diac'])
            corrected[original_word_idx] = fixed_word
            reports.append((tokens[original_word_idx], fixed_word, True))
        else:
            reports.append((tokens[original_word_idx], None, False))
    return corrected, reports

fully_corrected_tokens, grammar_reports = correct_grammar(spelling_corrected_tokens, word_infos, grammar_issues)

for original, fixed, ok in grammar_reports:
    print(f"  '{original}'  ->  '{fixed}'" if ok else f"  '{original}'  -> could not auto-generate a fix")
print("\nText after spelling + grammar correction:")
print(' '.join(fully_corrected_tokens))

  'الجديد'  ->  'الجديدة'
  'تحبان'  ->  'تحب'
  'تقرأه'  ->  'تقران'

Text after spelling + grammar correction:
ذهبت الطالبة الجديدة إلى المدرسة صباحا . هي تحب القراءة كثيرا ، ولديها كتاب جميلان تقران كل يوم .


## Step 8 — Normalization

Canonicalizes each word's orthography — the standard normalization steps
used across Arabic NLP (search/IR pipelines and ML preprocessing alike):

- strip elongation (tatweel, `ـ`)
- strip diacritics (tashkeel / harakat)
- unify alef variants (أ إ آ ٱ → ا)
- alef maksura → yeh (ى → ي)
- teh marbuta → heh (ة → ه)

This intentionally runs **after** spelling/grammar correction, on already-
correct words — normalization is about canonical form, not about fixing
mistakes.

In [12]:
def normalize_word(word):
    w = araby.strip_tatweel(word)
    w = dediac_ar(w)
    w = normalize_alef_ar(w)
    w = normalize_alef_maksura_ar(w)
    w = normalize_teh_marbuta_ar(w)
    return w

def normalize_text(tokens):
    return [normalize_word(t) if is_arabic_token(t) else t for t in tokens]

normalized_tokens = normalize_text(fully_corrected_tokens)

for before, after in zip(fully_corrected_tokens, normalized_tokens):
    marker = "  <- changed" if before != after else ""
    print(f"  {before:12s} -> {after}{marker}")

  ذهبت         -> ذهبت
  الطالبة      -> الطالبه  <- changed
  الجديدة      -> الجديده  <- changed
  إلى          -> الي  <- changed
  المدرسة      -> المدرسه  <- changed
  صباحاً       -> صباحا  <- changed
  .            -> .
  هي           -> هي
  تحب          -> تحب
  القراءة      -> القراءه  <- changed
  كثيراً       -> كثيرا  <- changed
  ،            -> ،
  وعندها       -> وعندها
  كتاب         -> كتاب
  جميل         -> جميل
  تقرأه        -> تقراه  <- changed
  كل           -> كل
  يوم          -> يوم
  .            -> .


## Step 9 — Stemming

**Tashaphyne** light-stems each word: it strips a known list of Arabic
prefixes/suffixes using a finite-state automaton and returns the remaining
stem (and, as a bonus, its own guess at the triliteral/quadriliteral root).
Light stemming is a *surface-pattern* technique — fast and dependency-free,
but it doesn't validate against a dictionary the way the morphological
analyzer does, so it can occasionally over-strip a word whose first letter
or two happen to look like a common prefix. That's a known, published
characteristic of light stemmers in general, not specific to this word list.

In [13]:
def stem_text(tokens):
    stems, roots = [], []
    for t in tokens:
        if is_arabic_token(t):
            light_stemmer.light_stem(t)
            stems.append(light_stemmer.get_stem())
            roots.append(light_stemmer.get_root())
        else:
            stems.append(t)
            roots.append(t)
    return stems, roots

stems, roots = stem_text(fully_corrected_tokens)

for word, stem, root in zip(fully_corrected_tokens, stems, roots):
    if is_arabic_token(word):
        print(f"  {word:12s} stem={stem:10s} root={root}")

  ذهبت         stem=ذهب        root=ذهب
  الطالبة      stem=طالب       root=طلب
  الجديدة      stem=جديد       root=جدد
  إلى          stem=إلى        root=إلى
  المدرسة      stem=مدرس       root=درس
  صباحاً       stem=صباحا      root=صباحا
  هي           stem=هي         root=هي
  تحب          stem=حب         root=حبب
  القراءة      stem=قراء       root=قرء
  كثيراً       stem=ثيرا       root=ثيرا
  وعندها       stem=عند        root=عند
  كتاب         stem=تاب        root=توب
  جميل         stem=جميل       root=جمل
  تقرأه        stem=قرأ        root=قرء
  كل           stem=كل         root=كل
  يوم          stem=وم         root=يوم


## Step 10 — Lemma analysis

The **lemma** (dictionary/citation form) is read straight from the MLE
disambiguator's top analysis for each word — the same tool used for grammar
detection in Step 4, re-run here on the fully corrected text so the lemmas
reflect the corrected words rather than the original typo/mismatch. CAMeL
lemmas carry a homograph index suffix (e.g. `جَدِيد_1`) to distinguish
same-spelled-different-meaning entries; that index is stripped for display,
along with diacritics, since most downstream uses want the plain form.

This is genuinely different from stemming: the lemma is a real, valid,
fully-inflectable dictionary word looked up via morphological analysis,
whereas a stem is just "whatever's left after cutting off known affixes" and
isn't necessarily a word on its own (see `تحب` -> stem `حب`, lemma `أحب`
below — `حب` is a truncated stem, `أحب` is the actual verb "to love").

In [14]:
def lemmatize_text(tokens):
    infos = analyze_words_in_context(tokens)
    lemmas = []
    for info in infos:
        lex = info.get('lex')
        if lex:
            lemma_clean = re.sub(r'_\d+$', '', lex)
            lemmas.append(dediac_ar(lemma_clean))
        else:
            lemmas.append(info['word'])
    return lemmas

lemmas = lemmatize_text(fully_corrected_tokens)

for word, stem, lemma in zip(fully_corrected_tokens, stems, lemmas):
    if is_arabic_token(word):
        print(f"  {word:12s} stem={stem:10s} lemma={lemma}")

  ذهبت         stem=ذهب        lemma=ذهب
  الطالبة      stem=طالب       lemma=طالب
  الجديدة      stem=جديد       lemma=جديد
  إلى          stem=إلى        lemma=إلى
  المدرسة      stem=مدرس       lemma=مدرسة
  صباحاً       stem=صباحا      lemma=صباح
  هي           stem=هي         lemma=هي
  تحب          stem=حب         lemma=أحب
  القراءة      stem=قراء       lemma=قراءة
  كثيراً       stem=ثيرا       lemma=كثير
  وعندها       stem=عند        lemma=عند
  كتاب         stem=تاب        lemma=كتاب
  جميل         stem=جميل       lemma=جميل
  تقرأه        stem=قرأ        lemma=أقر
  كل           stem=كل         lemma=كل
  يوم          stem=وم         lemma=يوم


## Full pipeline

Everything above, wrapped into one function that runs start to finish on any
text and returns a structured report — plus a per-word table and the final
before/after view.

In [86]:
def render_highlighted_html(tokens, spelling_errors, grammar_issues):
    spelling_positions = {p for positions in spelling_errors.values() for p in positions}
    grammar_positions = {}
    for issue in grammar_issues:
        if issue['type'] == 'noun_adj':
            grammar_positions[issue['adj_idx']] = issue['explanation']
        elif issue['type'] == 'sub_verb':
            grammar_positions[issue['verb_idx']] = issue['explanation']

    spans = []
    for i, tok in enumerate(tokens):
        safe_tok = html.escape(tok)
        if i in spelling_positions:
            spans.append(
                f'<span style="background:#ffd6d6;border-bottom:2px solid #d33;'
                f'border-radius:3px;padding:1px 4px;" title="Spelling error">{safe_tok}</span>'
            )
        elif i in grammar_positions:
            spans.append(
                f'<span style="background:#ffe8c2;border-bottom:2px solid #e08a00;'
                f'border-radius:3px;padding:1px 4px;" '
                f'title="Grammar error: {html.escape(grammar_positions[i])}">{safe_tok}</span>'
            )
        else:
            spans.append(safe_tok)

    legend = (
        '<div style="margin-bottom:8px;font-family:sans-serif;font-size:13px;color:#444;">'
        '<span style="background:#ffd6d6;border-bottom:2px solid #d33;padding:1px 6px;'
        'border-radius:3px;">spelling error</span>&nbsp;&nbsp;'
        '<span style="background:#ffe8c2;border-bottom:2px solid #e08a00;padding:1px 6px;'
        'border-radius:3px;">grammar error</span></div>'
    )
    body = (
        f'<div dir="rtl" style="font-size:22px;line-height:2.4;'
        f'font-family:\'Traditional Arabic\',Tahoma,Arial,sans-serif;">{ " ".join(spans)}</div>'
    )
    display(HTML(legend + body))

def run_arabic_pipeline(text, show_output=True):
    report = {'input_text': text}

    arabic_check = check_is_arabic(text)
    report['arabic_check'] = arabic_check
    if not arabic_check['is_arabic']:
        if show_output:
            print(f"This does not look like Arabic text (Arabic character ratio "
                  f"{arabic_check['arabic_char_ratio']:.0%}). Stopping.")
        return report

    tokens = tokenize_arabic(text)
    spelling_errors = detect_spelling_errors(tokens)
    spelling_error_positions = {p for positions in spelling_errors.values() for p in positions}

    word_infos = analyze_words_in_context(tokens)

    grammar_issues = detect_grammar_errors(word_infos, spelling_error_positions)

    if show_output:
        print("Detected errors, highlighted on the original text:")
        render_highlighted_html(tokens, spelling_errors, grammar_issues)

    spelling_corrected_tokens, spelling_corrections = correct_spelling(tokens, spelling_errors)
    fully_corrected_tokens, grammar_reports = correct_grammar(spelling_corrected_tokens, word_infos, grammar_issues)

    normalized_tokens = normalize_text(fully_corrected_tokens)
    stems, roots = stem_text(fully_corrected_tokens)
    lemmas = lemmatize_text(fully_corrected_tokens)

    table = pd.DataFrame({
        'original': tokens,
        'corrected': fully_corrected_tokens,
        'normalized': normalized_tokens,
        'stem': stems,
        'root': roots,
        'lemma': lemmas,
    })

    report.update({
        'tokens': tokens,
        'spelling_errors': spelling_errors,
        'spelling_corrections': spelling_corrections,
        'grammar_issues': grammar_issues,
        'grammar_reports': grammar_reports,
        'corrected_text': ' '.join(fully_corrected_tokens),
        'table': table,
    })

    if show_output:
        print(f"\nCorrected text:\n{report['corrected_text']}")
        print("\nPer-word breakdown:")
        display(table)

    return report

_ = run_arabic_pipeline(input_text)

DEBUG: Checking Noun 'الطالبة' (idx=1, pos=noun, gen=f, num=s)
DEBUG: Adjective Candidate 'الجديد' (idx=2, pos=adj, gen=m, num=s)
DEBUG: Mismatch results: gender_mismatch=True, number_mismatch=False
DEBUG: Noun-Adjective agreement issue found for 'الطالبة' and 'الجديد'.
DEBUG-TARGET: Found 'كتابان' and 'جميل' at indices 13, 14
DEBUG-TARGET: current_word_info: {'index': 13, 'word': 'كتابان', 'pos': 'noun', 'gen': 'm', 'num': 'd', 'stt': 'i', 'cas': 'n', 'lex': 'كِتاب', 'prc0': '0'}
DEBUG-TARGET: next_word_info: {'index': 14, 'word': 'جميل', 'pos': 'noun_prop', 'gen': 'm', 'num': 's', 'stt': 'i', 'cas': 'u', 'lex': 'جَمِيل', 'prc0': '0'}
DEBUG-TARGET: analyzer.analyze('جميل'): [{'diac': 'جَمِيل', 'lex': 'جَمِيل', 'bw': 'جَمِيل/NOUN_PROP', 'gloss': 'Jameel;Jamil;Gameel', 'pos': 'noun_prop', 'prc3': '0', 'prc2': '0', 'prc1': '0', 'prc0': '0', 'per': 'na', 'asp': 'na', 'vox': 'na', 'mod': 'na', 'stt': 'i', 'cas': 'u', 'enc0': '0', 'rat': 'r', 'source': 'lex', 'form_gen': 'm', 'form_num': 's


Corrected text:
ذهبت الطالبة الجديدة إلى المدرسة صباحا . هي تحب القراءة كثيرا ، وعندها كتابان جميل تقرأه كل يوم .

Per-word breakdown:


,original,corrected,normalized,stem,root,lemma
0,ذهبت,ذهبت,ذهبت,ذهب,ذهب,ذهب
1,الطالبة,الطالبة,الطالبه,طالب,طلب,طالب
2,الجديد,الجديدة,الجديده,جديد,جدد,جديد
3,إلى,إلى,الي,إلى,إلى,إلى
4,المدرصة,المدرسة,المدرسه,مدرس,درس,مدرسة
5,ضباحاً,صباحا,صباحا,صباح,صبح,صباح
6,.,.,.,.,.,.
7,هي,هي,هي,هي,هي,هي
8,تحبان,تحب,تحب,حب,حبب,أحب
9,القراءة,القراءة,القراءه,قراء,قرء,قراءة


## Try it with your own text

Edit `my_text` below and re-run this one cell — no need to re-run Setup.
The seeded example flips the agreement error the other way round (masculine
noun + feminine adjective this time) and uses a fresh typo, just to show the
rules aren't hard-coded to one direction or one word.

In [ ]:
my_text = "اشترى الطالب قلماً جديدة من مكتزة قريبة."
_ = run_arabic_pipeline(my_text)

## Notes, scope, and known limitations

**What this notebook deliberately does *not* try to do:**

- **No LLMs anywhere.** Every step is a classical, inspectable technique —
  a DB-backed morphological analyzer, an MLE disambiguator, edit-distance
  spell checking ranked by corpus frequency, and rule-based agreement
  checking + morphological reinflection.
- **Grammar checking is agreement-only, and scoped to singular/dual nouns.**
  Real Arabic grammar checking (case/*iʿrāb* errors, verb-subject agreement
  across long distances, missing/extra prepositions, word order, broken-
  plural rationality-based agreement, etc.) is a much bigger problem — this
  implements one well-defined, high-precision rule rather than guessing at
  the rest. See the CAMeL Lab's "Automatic Error Type Annotation for Arabic"
  (ARETA) work for the fuller error taxonomy this rule borrows its framing
  from (gender/number/definiteness/orthographic categories).
- **Spelling correction quality depends on the frequency dictionary.**
  A 50k-word general list won't have every proper noun, technical term, or
  neologism — those will still be flagged as "errors" even when they're
  legitimate words the dictionary just doesn't contain. Swap in a larger
  list (e.g. CAMeL Lab's own frequency lists, linked below) for better
  domain coverage.
- **The MLE disambiguator scores each word independently** (no full
  sentence-level neural context), so POS/lemma choice can occasionally land
  on a less-likely-but-plausible reading for ambiguous forms — you may spot
  this in the lemma of `تقرأه` in the walkthrough above. A neural CAMeLBERT
  disambiguator exists and is more context-aware, at the cost of being much
  heavier to run.
- **Light stemming (Tashaphyne) can over-strip** words whose first letters
  coincidentally resemble a known prefix — a documented trait of light
  stemmers generally, not just this one.

**Further reading / alternative tools researched for this notebook:**
- CAMeL Tools docs: https://camel-tools.readthedocs.io
- CAMeL Lab Arabic frequency lists (bigger than the 50k list used here): https://github.com/CAMeL-Lab/Camel_Arabic_Frequency_Lists
- Qalsadi (alternative rule-based Arabic lemmatizer): https://github.com/linuxscout/qalsadi
- Farasa (alternative Arabic segmentation/POS toolkit, QCRI): https://farasa.qcri.org
- FrequencyWords (spelling frequency lists in 100+ languages): https://github.com/hermitdave/FrequencyWords